# Clase 22: Clasificación de Imágenes con MNIST

**Objetivo:** Aplicar lo aprendido (Logística, Random Forest, EDA, métricas) a un problema real de visión por computadora.

**Dataset:** MNIST (dígitos escritos a mano) y Fashion MNIST (prendas de ropa).

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 1. Cargar MNIST

In [ ]:
from sklearn.datasets import fetch_openml
import numpy as np

X_full, y_full = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False, parser="auto")

# Subset de 15K para velocidad
rng = np.random.default_rng(42)
idx = rng.choice(len(X_full), 15000, replace=False)
X, y = X_full[idx], y_full[idx]

print(f"Imágenes: {X.shape[0]}")
print(f"Pixeles por imagen: {X.shape[1]}")
print(f"Rango pixeles: {X.min()}-{X.max()}")
print(f"Clases: {sorted(set(y))}")

70K imágenes originales, usamos 15K para velocidad. Cada imagen: 28×28 = 784 pixeles. Cada pixel: 0 (negro) a 255 (blanco). 10 clases (dígitos 0-9).

## 2. Entender la estructura: de imagen a vector

In [ ]:
# Una imagen es un vector de 784 números
print(f"Vector: {X[0].shape}")  # (784,)
print(f"Imagen: {X[0].reshape(28, 28).shape}")  # (28, 28)

# Visualizar
fig, axes = plt.subplots(2, 10, figsize=(14, 3.5))
for digit in range(10):
    idxs = np.where(y.astype(int) == digit)[0]
    for row in range(2):
        ax = axes[row, digit]
        ax.imshow(X[idxs[row]].reshape(28, 28), cmap="gray_r")
        if row == 0:
            ax.set_title(str(digit), fontweight="bold", fontsize=13)
        ax.axis("off")
plt.suptitle("MNIST: dígitos escritos a mano (28×28 = 784 variables)", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

Cada fila de X es un vector de 784 números. Para verla como imagen: `.reshape(28, 28)`. Cada persona escribe distinto — el modelo tiene que aprender a pesar de la variabilidad.

## 3. EDA de imágenes

### 3.1 Distribución de clases

In [ ]:
unique, counts = np.unique(y, return_counts=True)
fig = px.bar(x=[str(u) for u in unique], y=counts, title="Distribución de clases",
    labels={"x": "Dígito", "y": "Cantidad"}, color_discrete_sequence=["#C82B40"])
fig.update_layout(template="plotly_white")
fig.show()

Clases balanceadas (~10% cada una). No hay desbalance.

### 3.2 Dígito promedio por clase

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for digit in range(10):
    mask = y.astype(int) == digit
    avg = X[mask].mean(axis=0).reshape(28, 28)
    axes[digit].imshow(avg, cmap="hot")
    axes[digit].set_title(str(digit), fontweight="bold", fontsize=13)
    axes[digit].axis("off")
plt.suptitle("Imagen PROMEDIO de cada dígito", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

El promedio muestra qué pixeles se activan SIEMPRE para cada dígito. El '0' activa un anillo, el '1' una línea vertical. Estos patrones son lo que el modelo aprende.

### 3.3 Varianza por pixel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
var_map = X.var(axis=0).reshape(28, 28)
axes[0].imshow(var_map, cmap="Reds")
axes[0].set_title("Varianza por pixel\n(rojo = mucha variación)", fontweight="bold")
axes[0].axis("off")

zero_pct = (X == 0).mean(axis=0).reshape(28, 28)
axes[1].imshow(zero_pct, cmap="Blues")
axes[1].set_title("% de veces que el pixel es 0\n(azul = siempre vacío)", fontweight="bold")
axes[1].axis("off")
plt.suptitle("No todos los pixeles son útiles", fontweight="bold", fontsize=12)
plt.tight_layout()
plt.show()

Las esquinas siempre están vacías (azul oscuro). Los pixeles del centro varían mucho (rojo). De 784 pixeles, muchos son inútiles.

### 3.4 Imágenes difíciles

In [ ]:
# Mostrar imágenes con alta varianza dentro de su clase (las más "raras")
fig, axes = plt.subplots(2, 10, figsize=(14, 3.5))
for digit in range(10):
    mask = y.astype(int) == digit
    X_digit = X[mask]
    avg = X_digit.mean(axis=0)
    dists = np.linalg.norm(X_digit - avg, axis=1)
    weirdest = np.argsort(dists)[-2:]  # 2 más raros
    for row, w in enumerate(weirdest):
        ax = axes[row, digit]
        ax.imshow(X_digit[w].reshape(28, 28), cmap="gray_r")
        if row == 0:
            ax.set_title(f"'{digit}' raro", fontsize=9, color="#C82B40", fontweight="bold")
        ax.axis("off")
plt.suptitle("Dígitos 'difíciles': los más diferentes del promedio de su clase", fontweight="bold", fontsize=12)
plt.tight_layout()
plt.show()

Estos son los dígitos que más se alejan del promedio de su clase. Un '1' inclinado, un '7' con barra extraña. Si un humano duda, el modelo también.

## 4. Preprocesamiento

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Normalizar: de [0, 255] a [0, 1]
X_train_n = X_train / 255.0
X_test_n = X_test / 255.0
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Rango normalizado: {X_train_n.min():.1f} - {X_train_n.max():.1f}")

Dividir entre 255 lleva todos los pixeles al rango [0, 1]. Es el estándar para imágenes. Logística lo necesita, RF no (pero no le hace daño).

## 5. Modelo 1: Regresión Logística

In [ ]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

t0 = time.time()
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_n, y_train)
t1 = time.time()

y_pred_lr = lr.predict(X_test_n)
acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"Accuracy: {acc_lr:.3f} ({t1-t0:.1f}s)")
print(classification_report(y_test, y_pred_lr))

### Confusion Matrix — Logística

In [ ]:
cm = confusion_matrix(y_test, y_pred_lr, labels=[str(i) for i in range(10)])
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, xticklabels=range(10), yticklabels=range(10))
ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
ax.set_title(f"Confusion Matrix — Logística ({acc_lr:.1%})", fontweight="bold")
plt.tight_layout()
plt.show()

~90% accuracy. Errores principales: 3<->8, 3<->5, 9<->7, 9<->4. Todos son dígitos visualmente similares.

### Análisis de errores visual

In [ ]:
wrong = y_pred_lr != y_test
wrong_idx = np.where(wrong)[0]

fig, axes = plt.subplots(2, 10, figsize=(14, 3.5))
for i in range(min(20, len(wrong_idx))):
    ax = axes[i // 10, i % 10]
    ax.imshow(X_test[wrong_idx[i]].reshape(28, 28), cmap="gray_r")
    ax.set_title(f"{y_test[wrong_idx[i]]}→{y_pred_lr[wrong_idx[i]]}", fontsize=10, color="#C82B40", fontweight="bold")
    ax.axis("off")
plt.suptitle("Errores: real → predicho (muchos son ambiguos para humanos también)", fontweight="bold", fontsize=12)
plt.tight_layout()
plt.show()

Los errores no son 'estúpidos'. Son dígitos que un humano también dudaría. El modelo no es tonto — los datos son ambiguos.

## 6. Modelo 2: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

t0 = time.time()
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
t1 = time.time()

y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print(f"Accuracy: {acc_rf:.3f} ({t1-t0:.1f}s)")

### Confusion Matrix — Random Forest

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf, labels=[str(i) for i in range(10)])
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Greens", ax=ax, xticklabels=range(10), yticklabels=range(10))
ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
ax.set_title(f"Confusion Matrix — Random Forest ({acc_rf:.1%})", fontweight="bold")
plt.tight_layout()
plt.show()

~95% accuracy. +5% sobre logística. RF mejora en los dígitos difíciles porque captura combinaciones no lineales entre pixeles. Pero 9<->4 sigue difícil.

### Comparación de modelos

In [ ]:
fig = px.bar(x=["Logística", "Random Forest"], y=[acc_lr, acc_rf],
    title="Comparación de modelos en MNIST",
    labels={"x": "Modelo", "y": "Accuracy"},
    color=["Logística", "Random Forest"],
    color_discrete_map={"Logística": "#2563EB", "Random Forest": "#16A34A"},
    text=[f"{acc_lr:.1%}", f"{acc_rf:.1%}"])
fig.update_layout(template="plotly_white", showlegend=False)
fig.show()

Pasar de 90% a 95% no es 'solo 5%'. Significa la mitad de errores. En 10,000 cartas son 500 menos mal enviadas.

### Ejercicio 1: Analizar errores de Random Forest

**Instrucciones:**
1. Compara los errores de Logística vs Random Forest: ¿cuántos errores corrigió RF?
2. Muestra las imágenes que RF TODAVIA clasifica mal.
3. Interpreta: ¿son errores razonables?

In [ ]:
# Tu código aquí


## 7. Gradio — Dibujar y predecir

In [ ]:
# !pip install gradio
import gradio as gr
from PIL import Image

def predecir_digito(imagen):
    img = Image.fromarray(imagen).convert("L").resize((28, 28))
    pixels = np.array(img).flatten() / 255.0
    pixels = 1 - pixels  # invertir (Gradio fondo blanco, MNIST fondo negro)
    pred = rf.predict((pixels * 255).reshape(1, -1))[0]
    proba = rf.predict_proba((pixels * 255).reshape(1, -1))[0]
    return {str(i): float(proba[i]) for i in range(10)}

demo = gr.Interface(
    fn=predecir_digito,
    inputs=gr.Sketchpad(type="numpy"),
    outputs=gr.Label(num_top_classes=5),
    title="Dibuja un dígito (0-9)")
demo.launch()

`gr.Sketchpad` permite dibujar con el mouse. `gr.Label` muestra probabilidades como barras. El modelo predice en tiempo real.

## 8. Fashion MNIST — El techo

In [ ]:
print("Cargando Fashion MNIST...")
X_f, y_f = fetch_openml("Fashion-MNIST", version=1, return_X_y=True, as_frame=False, parser="auto")
idx_f = rng.choice(len(X_f), 15000, replace=False)
X_f, y_f = X_f[idx_f], y_f[idx_f]

fashion_labels = ["Camiseta", "Pantalón", "Suéter", "Vestido", "Abrigo",
                  "Sandalia", "Camisa", "Zapatilla", "Bolso", "Bota"]

### Muestras de Fashion MNIST

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(14, 4))
for cls in range(10):
    idxs_f = np.where(y_f.astype(int) == cls)[0]
    for row in range(2):
        ax = axes[row, cls]
        ax.imshow(X_f[idxs_f[row]].reshape(28, 28), cmap="gray_r")
        if row == 0:
            ax.set_title(fashion_labels[cls], fontweight="bold", fontsize=9)
        ax.axis("off")
plt.suptitle("Fashion MNIST: misma estructura, MUCHO más difícil", fontweight="bold")
plt.tight_layout()
plt.show()

### Entrenar y comparar

In [ ]:
X_f_tr, X_f_te, y_f_tr, y_f_te = train_test_split(X_f, y_f, test_size=0.2, random_state=42, stratify=y_f)

lr_f = LogisticRegression(max_iter=1000, random_state=42)
lr_f.fit(X_f_tr / 255.0, y_f_tr)
acc_lr_f = accuracy_score(y_f_te, lr_f.predict(X_f_te / 255.0))

rf_f = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_f.fit(X_f_tr, y_f_tr)
acc_rf_f = accuracy_score(y_f_te, rf_f.predict(X_f_te))

print(f"Fashion — Logística: {acc_lr_f:.3f}, RF: {acc_rf_f:.3f}")
print(f"MNIST   — Logística: {acc_lr:.3f}, RF: {acc_rf:.3f}")

### Comparación MNIST vs Fashion MNIST

In [ ]:
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Bar(x=["MNIST", "Fashion"], y=[acc_lr, acc_lr_f], name="Logística", marker_color="#2563EB"))
fig.add_trace(go.Bar(x=["MNIST", "Fashion"], y=[acc_rf, acc_rf_f], name="Random Forest", marker_color="#16A34A"))
fig.update_layout(barmode="group", title="MNIST vs Fashion MNIST: el techo", template="plotly_white", yaxis_range=[0.75, 1.0])
fig.show()

Con ropa, la logística baja de 90% a 84% y RF de 95% a 87%. Este es el techo de nuestras herramientas lineales + RF. Para ir más allá: redes neuronales convolucionales (CNN).

### Ejercicio 2: Confusion Matrix de Fashion MNIST

**Instrucciones:**
1. Calcula la confusion matrix de Random Forest en Fashion MNIST.
2. Usa `fashion_labels` como etiquetas en los ejes.
3. Muestra las imágenes que el modelo confunde.
4. Interpreta: ¿qué prendas se confunden y por qué?

In [ ]:
# Tu código aquí


## 9. ¿Qué sigue?

Nuestros modelos tratan cada pixel como independiente. No saben que el pixel 100 está al lado del 101. Las **CNN** procesan la imagen como 2D, detectan bordes y formas.

- CNN en MNIST: **99.7%**
- Todo lo que aprendieron (EDA, train/test, métricas, confusion matrix, GridSearchCV) se reutiliza con CNNs.

## Resumen

| Concepto | Detalle |
|---|---|
| Dataset | MNIST: 70K imágenes 28x28, 10 clases (dígitos) |
| Preprocesamiento | Normalizar /255, train_test_split con stratify |
| Logística | ~90% accuracy, lineal, rápida |
| Random Forest | ~95% accuracy, no lineal, más lenta |
| Fashion MNIST | Mismo formato, mucho más difícil |
| Gradio | Interface interactiva para probar el modelo |
| Siguiente paso | CNN (redes convolucionales) para >99% |